## Introduction to FMA-medium

FMA-medium is a curated subset of the Free Music Archive (FMA) dataset built for music information retrieval (MIR) and ML tasks such as genre recognition, created to give researchers an openly usable collection of Creative Commons–licensed music with standardized splits, precomputed features, and rich, structured metadata—so experiments are reproducible without redoing all the dataset plumbing. In practice, FMA-medium provides 25,000 fixed-length (30s) MP3 clips, labeled into 16 unbalanced genres, with an audio archive around ~22 GiB, and it ships with ***fma_metadata*** tables (track/artist/album info, genre taxonomy, and recommended split) plus baseline references that make it ideal for benchmarking and iterative model development.


## fma_metadata

Below is what **each CSV in `fma_metadata`** means. (Most people see **4 core CSVs**; some downloads also include **raw_*.csv** and **raw_albums.csv**)

---

### The 4 core metadata CSVs (the ones the official notebooks use)

#### 1) `tracks.csv` — “one row per track” + joined artist/album fields + split info

**Primary key / join key:** `track_id` (it’s the index when loaded with `utils.load`).

**What it contains:** basically everything you need to *label*, *filter*, and *split* tracks:

* **Track-level fields**: title, duration, listen counts, favorites, comments, dates, language, etc.
* **Genre fields**:

  * `track.genre_top`: a **single top genre label** (commonly used as the classification target)
  * `track.genres` / `track.genres_all`: **multi-genre assignments** (as lists of genre IDs)
* **Tags**: `track.tags` (list)
* **Album + Artist fields** (embedded as columns): album title/listens/tags, artist name/location/tags, etc.
* **The crucial “set” columns** used for ML protocol:

  * `set.subset` (e.g., small/medium/large)
  * `set.split` (training / validation / test)

**Notebook reality check (from `analysis.ipynb`):**

* It uses `tracks['set','subset'] <= 'medium'` to select a subset.
* It uses `tracks['set','split']` to compute per-split genre counts and imbalance. It means the split definition is in metadata (tracks.csv is the source of truth)

**Gotcha:** this file typically has a **2-row header** (pandas MultiIndex columns like `('track','title')`, `('artist','name')`, `('set','split')`). `utils.load` is designed to read that correctly. ([GitHub][1])

---

#### 2) `genres.csv` — the genre taxonomy (a tree), plus track counts

**Primary key / join key:** `genre_id` (usually the index).

**What it contains:**

* `title`: genre name
* `parent`: parent genre id (0 or null means “top-level”)
* often helper columns like `top_level` (boolean/flag) and `#tracks` (how many tracks in that genre)

**How it’s used in the notebooks:**

* to map genre IDs → names
* to reconstruct the **hierarchy** and identify top-level genres
* to validate that `track.genre_top` matches the set of top-level genres ([GitHub][1])

---

#### 3) `features.csv` — precomputed audio features (librosa), summarized

**Primary key / join key:** `track_id` (same index as `tracks.csv`).

**What it contains:** numeric feature vectors computed from audio using **librosa** and then summarized over time. The dataset documentation describes **feature families** like **chroma, tonnetz, MFCC, spectral centroid/bandwidth/contrast/rolloff, RMS, zero-crossing rate**, each summarized with statistics such as mean/std/etc. ([GitHub][1])

**Shape/structure:**

* Usually also a **MultiIndex column header** (feature family → statistic → coefficient/bin).
* In the official notebook they assert `features.index == tracks.index` (so it lines up perfectly). ([GitHub][2])

**How to think about it:** this is the “ready-to-train classical ML table” (RandomForest / SVM / XGBoost etc.) without touching raw audio.

---

#### 4) `echonest.csv` — Echo Nest audio features for a subset of tracks

**Primary key / join key:** `track_id` (subset; not all tracks have rows here).

**What it contains:** extra audio descriptors (from Echo Nest / now Spotify lineage) for **~13k tracks**. The common columns people use/mention include things like **acousticness, danceability, energy, instrumentalness, liveness, speechiness, tempo, valence**. ([GitHub][1])

**Notebook reality check:**

* `analysis.ipynb` checks that `echonest.index` is a subset of `tracks.index` and prints how many are available.

**How to use it:** optional enrichment features (nice if you want “high-level musicality” variables), but you must handle missingness because coverage is partial. ([GitHub][1])

---

### “Why do some `fma_metadata` zips have more CSVs?”

Some distributions include **nine CSVs**, typically:

* the 4 processed ones above (`tracks.csv`, `genres.csv`, `features.csv`, `echonest.csv`)
* plus **raw_ versions** (e.g., `raw_tracks.csv`, `raw_features.csv`, `raw_genres.csv`, `raw_echonest.csv`)
* plus **`raw_albums.csv`** ([Ray Woodcock's Latest][3])

#### What the `raw_*.csv` files are

Think of them as **closer to the original scraped/source dumps** (messier / less standardized). The “non-raw” versions are the cleaned/parsed ones the official notebooks expect. Typical differences:

* list-like fields may be unparsed strings / JSON-ish blobs
* headers may not be the nice 2-row MultiIndex format
* IDs and joins may require extra cleaning

#### `raw_albums.csv`

A raw album table extracted separately (albums are otherwise “embedded” into `tracks.csv` as repeated album columns). It’s useful if you want to treat albums as first-class entities without deduplicating from `tracks.csv`. ([Ray Woodcock's Latest][3])

---

### Mental model 

* **`tracks.csv`** = *labels + splits + rich metadata* (the  “master table”). The key columns for this project are 'set' (split: training/validation/test) and 'set.1' (subset: small/medium/large)
* **`genres.csv`** = *label dictionary + hierarchy*
* **`features.csv`** = *audio → numbers (librosa), full coverage*
* **`echonest.csv`** = *audio → higher-level numbers, partial coverage*
* **`raw_*.csv`** = *archaeological layers* (useful, but usually not needed unless you’re rebuilding the cleaning pipeline)



### Metadata CSV quick samples 

The cell below locates `FMA/fma_metadata`, loads each CSV file, and displays exactly 9 sample rows from each file for data understanding.

In [12]:
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.expand_frame_repr", False)


def find_fma_metadata_dir(start_dir: Path | None = None) -> Path:
    base = (start_dir or Path.cwd()).resolve()
    for candidate_base in [base, *base.parents]:
        candidate = candidate_base / "FMA" / "fma_metadata"
        if candidate.exists() and candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find 'FMA/fma_metadata' from current working directory or its parents."
    )


metadata_dir = find_fma_metadata_dir()
csv_files = sorted(metadata_dir.glob("*.csv"))

print(f"Metadata folder: {metadata_dir}")
print(f"CSV files found: {len(csv_files)}")

for csv_path in csv_files:
    if not csv_path.name.startswith("raw"):
        print("\n" + "=" * 90)
        print(f"File: {csv_path.name}")

        header_df = pd.read_csv(csv_path, nrows=0)
        columns = header_df.columns.tolist()
        print(f"Columns ({len(columns)}):")
        print(columns)

        sample_df = pd.read_csv(csv_path, nrows=6)

        if csv_path.name == "tracks.csv":
            print("The key columns for this project are 'set' (split: training/validation/test) and 'set.1' (subset: small/medium/large)")
            artist_cols = [col for col in sample_df.columns if str(col).startswith("artist") or str(col).startswith("track")]
            sample_df = sample_df.copy()
            for col in artist_cols:
                sample_df[col] = sample_df[col].apply(
                    lambda value: value if pd.isna(value) else f"{str(value)[:9]}..."
                )

        print(f"Sample shape shown: {sample_df.shape}")
        display(sample_df)

Metadata folder: D:\mse\nguyen_sy_hung_codebases\machine-learning-1\FMA\fma_metadata
CSV files found: 9

File: echonest.csv
Columns (250):
['Unnamed: 0', 'echonest', 'echonest.1', 'echonest.2', 'echonest.3', 'echonest.4', 'echonest.5', 'echonest.6', 'echonest.7', 'echonest.8', 'echonest.9', 'echonest.10', 'echonest.11', 'echonest.12', 'echonest.13', 'echonest.14', 'echonest.15', 'echonest.16', 'echonest.17', 'echonest.18', 'echonest.19', 'echonest.20', 'echonest.21', 'echonest.22', 'echonest.23', 'echonest.24', 'echonest.25', 'echonest.26', 'echonest.27', 'echonest.28', 'echonest.29', 'echonest.30', 'echonest.31', 'echonest.32', 'echonest.33', 'echonest.34', 'echonest.35', 'echonest.36', 'echonest.37', 'echonest.38', 'echonest.39', 'echonest.40', 'echonest.41', 'echonest.42', 'echonest.43', 'echonest.44', 'echonest.45', 'echonest.46', 'echonest.47', 'echonest.48', 'echonest.49', 'echonest.50', 'echonest.51', 'echonest.52', 'echonest.53', 'echonest.54', 'echonest.55', 'echonest.56', 'ec

,Unnamed: 0,echonest,echonest.1,echonest.2,echonest.3,echonest.4,echonest.5,echonest.6,echonest.7,echonest.8,echonest.9,echonest.10,echonest.11,echonest.12,echonest.13,echonest.14,echonest.15,echonest.16,echonest.17,echonest.18,echonest.19,echonest.20,echonest.21,echonest.22,echonest.23,echonest.24,echonest.25,echonest.26,echonest.27,echonest.28,echonest.29,echonest.30,echonest.31,echonest.32,echonest.33,echonest.34,echonest.35,echonest.36,echonest.37,echonest.38,echonest.39,echonest.40,echonest.41,echonest.42,echonest.43,echonest.44,echonest.45,echonest.46,echonest.47,echonest.48,echonest.49,echonest.50,echonest.51,echonest.52,echonest.53,echonest.54,echonest.55,echonest.56,echonest.57,echonest.58,echonest.59,echonest.60,echonest.61,echonest.62,echonest.63,echonest.64,echonest.65,echonest.66,echonest.67,echonest.68,echonest.69,echonest.70,echonest.71,echonest.72,echonest.73,echonest.74,echonest.75,echonest.76,echonest.77,echonest.78,echonest.79,echonest.80,echonest.81,echonest.82,echonest.83,echonest.84,echonest.85,echonest.86,echonest.87,echonest.88,echonest.89,echonest.90,echonest.91,echonest.92,echonest.93,echonest.94,echonest.95,echonest.96,echonest.97,echonest.98,echonest.99,echonest.100,echonest.101,echonest.102,echonest.103,echonest.104,echonest.105,echonest.106,echonest.107,echonest.108,echonest.109,echonest.110,echonest.111,echonest.112,echonest.113,echonest.114,echonest.115,echonest.116,echonest.117,echonest.118,echonest.119,echonest.120,echonest.121,echonest.122,echonest.123,echonest.124,echonest.125,echonest.126,echonest.127,echonest.128,echonest.129,echonest.130,echonest.131,echonest.132,echonest.133,echonest.134,echonest.135,echonest.136,echonest.137,echonest.138,echonest.139,echonest.140,echonest.141,echonest.142,echonest.143,echonest.144,echonest.145,echonest.146,echonest.147,echonest.148,echonest.149,echonest.150,echonest.151,echonest.152,echonest.153,echonest.154,echonest.155,echonest.156,echonest.157,echonest.158,echonest.159,echonest.160,echonest.161,echonest.162,echonest.163,echonest.164,echonest.165,echonest.166,echonest.167,echonest.168,echonest.169,echonest.170,echonest.171,echonest.172,echonest.173,echonest.174,echonest.175,echonest.176,echonest.177,echonest.178,echonest.179,echonest.180,echonest.181,echonest.182,echonest.183,echonest.184,echonest.185,echonest.186,echonest.187,echonest.188,echonest.189,echonest.190,echonest.191,echonest.192,echonest.193,echonest.194,echonest.195,echonest.196,echonest.197,echonest.198,echonest.199,echonest.200,echonest.201,echonest.202,echonest.203,echonest.204,echonest.205,echonest.206,echonest.207,echonest.208,echonest.209,echonest.210,echonest.211,echonest.212,echonest.213,echonest.214,echonest.215,echonest.216,echonest.217,echonest.218,echonest.219,echonest.220,echonest.221,echonest.222,echonest.223,echonest.224,echonest.225,echonest.226,echonest.227,echonest.228,echonest.229,echonest.230,echonest.231,echonest.232,echonest.233,echonest.234,echonest.235,echonest.236,echonest.237,echonest.238,echonest.239,echonest.240,echonest.241,echonest.242,echonest.243,echonest.244,echonest.245,echonest.246,echonest.247,echonest.248
0,NaN,audio_features,audio_features,audio_features,audio_features,audio_features,audio_features,audio_features,audio_features,metadata,metadata,metadata,metadata,metadata,metadata,metadata,ranks,ranks,ranks,ranks,ranks,social_features,social_features,social_features,social_features,social_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,temporal_features,tempor


File: features.csv
Columns (519):
['feature', 'chroma_cens', 'chroma_cens.1', 'chroma_cens.2', 'chroma_cens.3', 'chroma_cens.4', 'chroma_cens.5', 'chroma_cens.6', 'chroma_cens.7', 'chroma_cens.8', 'chroma_cens.9', 'chroma_cens.10', 'chroma_cens.11', 'chroma_cens.12', 'chroma_cens.13', 'chroma_cens.14', 'chroma_cens.15', 'chroma_cens.16', 'chroma_cens.17', 'chroma_cens.18', 'chroma_cens.19', 'chroma_cens.20', 'chroma_cens.21', 'chroma_cens.22', 'chroma_cens.23', 'chroma_cens.24', 'chroma_cens.25', 'chroma_cens.26', 'chroma_cens.27', 'chroma_cens.28', 'chroma_cens.29', 'chroma_cens.30', 'chroma_cens.31', 'chroma_cens.32', 'chroma_cens.33', 'chroma_cens.34', 'chroma_cens.35', 'chroma_cens.36', 'chroma_cens.37', 'chroma_cens.38', 'chroma_cens.39', 'chroma_cens.40', 'chroma_cens.41', 'chroma_cens.42', 'chroma_cens.43', 'chroma_cens.44', 'chroma_cens.45', 'chroma_cens.46', 'chroma_cens.47', 'chroma_cens.48', 'chroma_cens.49', 'chroma_cens.50', 'chroma_cens.51', 'chroma_cens.52', 'chroma_cen

,feature,chroma_cens,chroma_cens.1,chroma_cens.2,chroma_cens.3,chroma_cens.4,chroma_cens.5,chroma_cens.6,chroma_cens.7,chroma_cens.8,chroma_cens.9,chroma_cens.10,chroma_cens.11,chroma_cens.12,chroma_cens.13,chroma_cens.14,chroma_cens.15,chroma_cens.16,chroma_cens.17,chroma_cens.18,chroma_cens.19,chroma_cens.20,chroma_cens.21,chroma_cens.22,chroma_cens.23,chroma_cens.24,chroma_cens.25,chroma_cens.26,chroma_cens.27,chroma_cens.28,chroma_cens.29,chroma_cens.30,chroma_cens.31,chroma_cens.32,chroma_cens.33,chroma_cens.34,chroma_cens.35,chroma_cens.36,chroma_cens.37,chroma_cens.38,chroma_cens.39,chroma_cens.40,chroma_cens.41,chroma_cens.42,chroma_cens.43,chroma_cens.44,chroma_cens.45,chroma_cens.46,chroma_cens.47,chroma_cens.48,chroma_cens.49,chroma_cens.50,chroma_cens.51,chroma_cens.52,chroma_cens.53,chroma_cens.54,chroma_cens.55,chroma_cens.56,chroma_cens.57,chroma_cens.58,chroma_cens.59,chroma_cens.60,chroma_cens.61,chroma_cens.62,chroma_cens.63,chroma_cens.64,chroma_cens.65,chroma_cens.66,chroma_cens.67,chroma_cens.68,chroma_cens.69,chroma_cens.70,chroma_cens.71,chroma_cens.72,chroma_cens.73,chroma_cens.74,chroma_cens.75,chroma_cens.76,chroma_cens.77,chroma_cens.78,chroma_cens.79,chroma_cens.80,chroma_cens.81,chroma_cens.82,chroma_cens.83,chroma_cqt,chroma_cqt.1,chroma_cqt.2,chroma_cqt.3,chroma_cqt.4,chroma_cqt.5,chroma_cqt.6,chroma_cqt.7,chroma_cqt.8,chroma_cqt.9,chroma_cqt.10,chroma_cqt.11,chroma_cqt.12,chroma_cqt.13,chroma_cqt.14,chroma_cqt.15,chroma_cqt.16,chroma_cqt.17,chroma_cqt.18,chroma_cqt.19,chroma_cqt.20,chroma_cqt.21,chroma_cqt.22,chroma_cqt.23,chroma_cqt.24,chroma_cqt.25,chroma_cqt.26,chroma_cqt.27,chroma_cqt.28,chroma_cqt.29,chroma_cqt.30,chroma_cqt.31,chroma_cqt.32,chroma_cqt.33,chroma_cqt.34,chroma_cqt.35,chroma_cqt.36,chroma_cqt.37,chroma_cqt.38,chroma_cqt.39,chroma_cqt.40,chroma_cqt.41,chroma_cqt.42,chroma_cqt.43,chroma_cqt.44,chroma_cqt.45,chroma_cqt.46,chroma_cqt.47,chroma_cqt.48,chroma_cqt.49,chroma_cqt.50,chroma_cqt.51,chroma_cqt.52,chroma_cqt.53,chroma_cqt.54,chroma_cqt.55,chroma_cqt.56,chroma_cqt.57,chroma_cqt.58,chroma_cqt.59,chroma_cqt.60,chroma_cqt.61,chroma_cqt.62,chroma_cqt.63,chroma_cqt.64,chroma_cqt.65,chroma_cqt.66,chroma_cqt.67,chroma_cqt.68,chroma_cqt.69,chroma_cqt.70,chroma_cqt.71,chroma_cqt.72,chroma_cqt.73,chroma_cqt.74,chroma_cqt.75,chroma_cqt.76,chroma_cqt.77,chroma_cqt.78,chroma_cqt.79,chroma_cqt.80,chroma_cqt.81,chroma_cqt.82,chroma_cqt.83,chroma_stft,chroma_stft.1,chroma_stft.2,chroma_stft.3,chroma_stft.4,chroma_stft.5,chroma_stft.6,chroma_stft.7,chroma_stft.8,chroma_stft.9,chroma_stft.10,chroma_stft.11,chroma_stft.12,chroma_stft.13,chroma_stft.14,chroma_stft.15,chroma_stft.16,chroma_stft.17,chroma_stft.18,chroma_stft.19,chroma_stft.20,chroma_stft.21,chroma_stft.22,chroma_stft.23,chroma_stft.24,chroma_stft.25,chroma_stft.26,chroma_stft.27,chroma_stft.28,chroma_stft.29,chroma_stft.30,chroma_stft.31,chroma_stft.32,chroma_stft.33,chroma_stft.34,chroma_stft.35,chroma_stft.36,chroma_stft.37,chroma_stft.38,chroma_stft.39,chroma_stft.40,chroma_stft.41,chroma_stft.42,chroma_stft.43,chroma_stft.44,chroma_stft.45,chroma_stft.46,chroma_stft.47,chroma_stft.48,chroma_stft.49,chroma_stft.50,chroma_stft.51,chroma_stft.52,chroma_stft.53,chroma_stft.54,chroma_stft.55,chroma_stft.56,chroma_stft.57,chroma_stft.58,chroma_stft.59,chroma_stft.60,chroma_stft.61,chroma_stft.62,chroma_stft.63,chroma_stft.64,chroma_stft.65,chroma_stft.66,chroma_stft.67,chroma_stft.68,chroma_stft.69,chroma_stft.70,chroma_stft.71,chroma_stft.72,chroma_stft.73,chroma_stft.74,chroma_stft.75,chroma_stft.76,chroma_stft.77,chroma_stft.78,chroma_stft.79,chroma_stft.80,chroma_stft.81,chroma_stft.82,chroma_stft.83,mfcc,mfcc.1,mfcc.2,mfcc.3,mfcc.4,mfcc.5,mfcc.6,mfcc.7,mfcc.8,mfcc.9,mfcc.10,mfcc.11,mfcc.12,mfcc.13,mfcc.14,mfcc.15,mfcc.16,mfcc.17,mfcc.18,mfcc.19,mfcc.20,mfcc.21,mfcc.22,mfcc.23,mfcc.24,mfcc.25,mfcc.26,mfcc.27,mfcc.28,mfcc.29,mfcc.30,mfcc.31,mfcc.32,mfcc.33,mfcc.34,mfcc.35,mfcc.36,mfcc.37,mfcc.38,mfcc.39,mfcc.40,mfcc.41,mfcc.42


File: genres.csv
Columns (5):
['genre_id', '#tracks', 'parent', 'title', 'top_level']
Sample shape shown: (6, 5)


,genre_id,#tracks,parent,title,top_level
0,1,8693,38,Avant-Garde,38
1,2,5271,0,International,2
2,3,1752,0,Blues,3
3,4,4126,0,Jazz,4
4,5,4106,0,Classical,5
5,6,914,38,Novelty,38



File: tracks.csv
Columns (53):
['Unnamed: 0', 'album', 'album.1', 'album.2', 'album.3', 'album.4', 'album.5', 'album.6', 'album.7', 'album.8', 'album.9', 'album.10', 'album.11', 'album.12', 'artist', 'artist.1', 'artist.2', 'artist.3', 'artist.4', 'artist.5', 'artist.6', 'artist.7', 'artist.8', 'artist.9', 'artist.10', 'artist.11', 'artist.12', 'artist.13', 'artist.14', 'artist.15', 'artist.16', 'set', 'set.1', 'track', 'track.1', 'track.2', 'track.3', 'track.4', 'track.5', 'track.6', 'track.7', 'track.8', 'track.9', 'track.10', 'track.11', 'track.12', 'track.13', 'track.14', 'track.15', 'track.16', 'track.17', 'track.18', 'track.19']
The key columns for this project are 'set' (split: training/validation/test) and 'set.1' (subset: small/medium/large)
Sample shape shown: (6, 53)


,Unnamed: 0,album,album.1,album.2,album.3,album.4,album.5,album.6,album.7,album.8,album.9,album.10,album.11,album.12,artist,artist.1,artist.2,artist.3,artist.4,artist.5,artist.6,artist.7,artist.8,artist.9,artist.10,artist.11,artist.12,artist.13,artist.14,artist.15,artist.16,set,set.1,track,track.1,track.2,track.3,track.4,track.5,track.6,track.7,track.8,track.9,track.10,track.11,track.12,track.13,track.14,track.15,track.16,track.17,track.18,track.19
0,NaN,comments,date_created,date_released,engineer,favorites,id,information,listens,producer,tags,title,tracks,type,active_ye...,active_ye...,associate...,bio...,comments...,date_crea...,favorites...,id...,latitude...,location...,longitude...,members...,name...,related_p...,tags...,website...,wikipedia...,split,subset,bit_rate...,comments...,composer...,date_crea...,date_reco...,duration...,favorites...,genre_top...,genres...,genres_al...,informati...,interest...,language_...,license...,listens...,lyricist...,number...,publisher...,tags...,title...
1,track_id,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,0,2008-11-26 01:44:45,2009-01-05 00:00:00,NaN,4,1,<p></p>,6073,NaN,[],AWOL - A Way Of Life,7,Album,2006-01-0...,NaN,NaN,<p>A Way ...,0...,2008-11-2...,9...,1...,40.058323...,New Jerse...,-74.40566...,Sajje Mor...,AWOL...,The list ...,['awol']...,http://ww...,NaN,training,small,256000...,0...,NaN,2008-11-2...,2008-11-2...,168...,2...,Hip-Hop...,[21]...,[21]...,NaN,4656...,en...,Attributi...,1293...,NaN,3...,NaN,[]...,Food...
3,3,0,2008-11-26 01:44:45,2009-01-05 00:00:00,NaN,4,1,<p></p>,6073,NaN,[],AWOL - A Way Of Life,7,Album,2006-01-0...,NaN,NaN,<p>A Way ...,0...,2008-11-2...,9...,1...,40.058323...,New Jerse...,-74.40566...,Sajje Mor...,AWOL...,The list ...,['awol']...,http://ww...,NaN,training,medium,256000...,0...,NaN,2008-11-2...,2008-11-2...,237...,1...,Hip-Hop...,[21]...,[21]...,NaN,1470...,en...,Attributi...,514...,NaN,4...,NaN,[]...,Electric ...
4,5,0,2008-11-26 01:44:45,2009-01-05 00:00:00,NaN,4,1,<p></p>,6073,NaN,[],AWOL - A Way Of Life,7,Album,2006-01-0...,NaN,NaN,<p>A Way ...,0...,2008-11-2...,9...,1...,40.058323...,New Jerse...,-74.40566...,Sajje Mor...,AWOL...,The list ...,['awol']...,http://ww...,NaN,training,small,256000...,0...,NaN,2008-11-2...,2008-11-2...,206...,6...,Hip-Hop...,[21]...,[21]...,NaN,1933...,en...,Attributi...,1151...,NaN,6...,NaN,[]...,This Worl...
5,10,0,2008-11-26 01:45:08,2008-02-06 00:00:00,NaN,4,6,NaN,47632,NaN,[],Constant Hitmaker,2,Album,NaN,NaN,Mexican S...,<p><span ...,3...,2008-11-2...,74...,6...,NaN,NaN,NaN,Kurt Vile...,Kurt Vile...,NaN,['philly'...,http://ku...,NaN,training,small,192000...,0...,Kurt Vile...,2008-11-2...,2008-11-2...,161...,178...,Pop...,[10]...,[10]...,NaN,54881...,en...,Attributi...,50135...,NaN,1...,NaN,[]...,Freeway...


---

## Metadata ↔ Audio File Mismatch Check

For each FMA audio subset (`small`, `medium`, `large`) that exists under `FMA/`, this section
compares the **track IDs declared in `tracks.csv`** against the **MP3 files actually on disk**
and reports three numbers:

| Metric | Meaning |
|---|---|
| Expected (metadata) | track IDs in `tracks.csv` with `set.subset == <name>` |
| Found on disk | `.mp3` files present in the audio folder |
| **Missing audio** | in metadata but no file → would cause a `FileNotFoundError` at training time |
| **Orphan files** | file exists but not declared in metadata → unreferenced audio |

Subsets whose folder does not exist under `FMA/` are silently skipped.

In [1]:
from pathlib import Path
import pandas as pd

# ── Locate workspace root from the notebook directory ─────────────────────────
NOTEBOOK_DIR = Path().resolve()              # …/MelCNN-MGR/notebooks
FMA_ROOT     = NOTEBOOK_DIR.parent.parent / "FMA"   # …/FMA

# ── Load tracks.csv (2-level header, track_id as index) ─────────────────────
tracks_csv = FMA_ROOT / "fma_metadata" / "tracks.csv"
if not tracks_csv.exists():
    raise FileNotFoundError(f"tracks.csv not found at {tracks_csv}")

tracks = pd.read_csv(tracks_csv, index_col=0, header=[0, 1])
tracks.index.name = "track_id"

# Flatten column used for subset membership
subset_col = ("set", "subset")


def get_expected_ids_for_subset(tracks: pd.DataFrame, subset_name: str) -> set:
    """Return the set of track_ids declared in tracks.csv for a given subset."""
    mask = tracks[subset_col] == subset_name
    return set(tracks.index[mask].tolist())


def get_disk_ids_for_audio_root(audio_root: Path) -> set:
    """Walk the audio folder and collect all track_ids inferred from .mp3 filenames.

    FMA convention: <audio_root>/<NNN>/<NNNNNN>.mp3
    Track id = int(stem), e.g. '012345.mp3' → 12345
    """
    return {int(p.stem) for p in audio_root.rglob("*.mp3")}


# ── Check each subset ────────────────────────────────────────────────────────
SUBSETS = ["small", "medium", "large"]
SEP     = "=" * 70

print(f"Metadata root : {FMA_ROOT / 'fma_metadata'}")
print(f"FMA root      : {FMA_ROOT}\n")

for subset_name in SUBSETS:
    audio_root = FMA_ROOT / f"fma_{subset_name}"

    if not audio_root.exists():
        print(f"{SEP}")
        print(f"  Subset : {subset_name}")
        print(f"  Audio folder not found — skipping  ({audio_root})")
        print()
        continue

    print(SEP)
    print(f"  Subset      : {subset_name}")
    print(f"  Audio root  : {audio_root}")

    expected_ids = get_expected_ids_for_subset(tracks, subset_name)
    print(f"  Scanning audio files … (this may take a moment on slow filesystems)")
    disk_ids     = get_disk_ids_for_audio_root(audio_root)

    missing_audio  = expected_ids - disk_ids    # in metadata, no file
    orphan_files   = disk_ids - expected_ids    # file exists, not in metadata

    print(f"  Expected (metadata)   : {len(expected_ids):>6,}")
    print(f"  Found on disk         : {len(disk_ids):>6,}")
    print()

    if not missing_audio:
        print("  ✓ Missing audio       :      0  — all expected files are present")
    else:
        print(f"  ✗ Missing audio       : {len(missing_audio):>6,}  — in metadata but no .mp3 on disk")
        sample = sorted(missing_audio)[:10]
        print(f"    First up-to-10 missing track_ids: {sample}"
              + (" …" if len(missing_audio) > 10 else ""))

    if not orphan_files:
        print("  ✓ Orphan files        :      0  — no unreferenced audio files")
    else:
        print(f"  ✗ Orphan files        : {len(orphan_files):>6,}  — .mp3 files not in metadata")
        sample = sorted(orphan_files)[:10]
        print(f"    First up-to-10 orphan track_ids: {sample}"
              + (" …" if len(orphan_files) > 10 else ""))

    print()

print(SEP)

Metadata root : /mnt/d/mse/nguyen_sy_hung_codebases/machine-learning-1/FMA/fma_metadata
FMA root      : /mnt/d/mse/nguyen_sy_hung_codebases/machine-learning-1/FMA

  Subset      : small
  Audio root  : /mnt/d/mse/nguyen_sy_hung_codebases/machine-learning-1/FMA/fma_small
  Scanning audio files … (this may take a moment on slow filesystems)
  Expected (metadata)   :  8,000
  Found on disk         :  8,000

  ✓ Missing audio       :      0  — all expected files are present
  ✓ Orphan files        :      0  — no unreferenced audio files

  Subset      : medium
  Audio root  : /mnt/d/mse/nguyen_sy_hung_codebases/machine-learning-1/FMA/fma_medium
  Scanning audio files … (this may take a moment on slow filesystems)
  Expected (metadata)   : 17,000
  Found on disk         : 25,000

  ✓ Missing audio       :      0  — all expected files are present
  ✗ Orphan files        :  8,000  — .mp3 files not in metadata
    First up-to-10 orphan track_ids: [2, 5, 10, 140, 141, 148, 182, 190, 193, 194] …

---

# FMA-medium labels (geners) and where to find them

For **FMA-medium**, the standard benchmark label is a **single genre per clip**, chosen from **16 top-level genres**. “Unbalanced” means those 16 classes don’t have equal track counts, so you should expect skewed distributions (and handle that in evaluation).


## 1) Where to find the genre label for a certain clip

### The source of truth: `fma_metadata/tracks.csv`

**In short, track.genre_top column in tracks.csv**

Each audio clip corresponds to a **track ID** (the numeric ID used in the audio filename path). To find the label:

1. Open **`fma_metadata/tracks.csv`**
2. Locate the row with **index = `track_id`**
3. Read the label columns:

* **`track.genre_top`** → the **primary single label** used for the 16-genre medium task
* **`track.genres` / `track.genres_all`** → the **full set of assigned genres**, stored as **genre IDs** (these are finer-grained and can include subgenres)

### Turning genre IDs into names: `fma_metadata/genres.csv`

If you use `track.genres_all` (IDs), map them via:

* `genres.csv`: **`genre_id` → `title`**

### Extra checks you’ll often want

In the same `tracks.csv` row:

* **`set.subset`**: confirm it’s `"medium"`
* **`set.split`**: tells you `"training"`, `"validation"`, or `"test"` (important for leakage-safe evaluation)

### Common “gotcha”

`tracks.csv` uses a **two-row header** (a pandas MultiIndex). If you load it “naively,” list-like columns (genres/tags) may appear as strings like `"[21, 38]"`. The official loader (`utils.load`) parses these properly; otherwise parse them yourself (e.g., `ast.literal_eval`) before mapping IDs.

---

## 2) Are the 16 “medium” genres the top-level genres in `genres.csv`?

**Yes, conceptually.** The “16 genres” used for FMA-medium classification correspond to the dataset’s **top-level (root) genres** in the genre taxonomy stored in `genres.csv`.

### How that’s represented in the tables

* The per-track label used for the medium benchmark is **`tracks.csv → track.genre_top`**.
* That label is derived from the genre hierarchy in **`genres.csv`**, collapsing finer-grained genre assignments to their **top-level ancestor**.

### What this implies

* `genres.csv` contains **many more than 16** genres (subgenres, sub-subgenres).
* The “16” is the **coarsest layer**, not the full taxonomy.
* So: **medium label = top-level genre name**, while `genres_all` can contain deeper taxonomy IDs.

### Quick sanity check (recommended)

To verify your local copy matches this expectation:

1. Filter to medium tracks using `set.subset == "medium"`
2. Compute `unique(track.genre_top)` → should be **16 labels**
3. Confirm those labels correspond to root genres in `genres.csv` (e.g., rows where `parent` is null/0, depending on your file version)

---

## Practical workflow summary

* Want the **standard label for training a 16-class classifier**? Use **`track.genre_top`**.
* Want the **full taxonomy assignment(s)** for analysis or hierarchical models? Use **`track.genres_all`** and map IDs using **`genres.csv`**.
* Always respect **`set.split`** and **`set.subset`** from `tracks.csv` when doing ML experiments.


## Genre Distribution Analysis

For a detailed split and genre distribution analysis, see [Data-Understanding-Train-Val-Test-Genre-Distribution.ipynb](./Data-Understanding-Train-Val-Test-Genre-Distribution.ipynb).